In [1]:
import pandas as pd
import os
import sys

In [ ]:
SSD_path = '../../../../../Volumes/SSD_01/'

mrkr_data_path ='../../../../../Volumes/HDD_02/datasets/emory_mrkr/'
mrkr_data_path = os.path.abspath(mrkr_data_path)
mrkr_tables_path = os.path.join(mrkr_data_path,'tables')

table_names = os.listdir(mrkr_tables_path)

image_df = pd.read_csv(
    os.path.join(
        mrkr_tables_path,
        table_names[-3]
    )
)

demo_df = pd.read_csv(
    os.path.join(
        mrkr_tables_path,
        table_names[-4]
    )
)

lateral_master_csv = os.path.join(mrkr_data_path, 'tables', 'lateral_master_df.csv')

if os.path.exists(lateral_master_csv):
    lateral_master_df = pd.read_csv(lateral_master_csv)
else:
    # build from image_df as before
    lateral_master_df = image_df[image_df.view_position == 'L'].reset_index(drop=True)
    lateral_master_df['mask_path'] = lateral_master_df['dicom_path'].apply(build_mask_path)
    lateral_master_df['is_segmented'] = False
    lateral_master_df.to_csv(lateral_master_csv, index=False)


In [ ]:
table_names

['MRKR_ICD_dictionary.csv',
 'MRKR_pain.csv',
 'MRKR_ICD.csv',
 'MRKR_demographics.csv',
 'MRKR_image_metadata.csv',
 'MRKR_CPT_dictionary.csv',
 'MRKR_CPT.csv']

In [12]:
img_df.head()

,empi_anon,StudyInstanceUID_anon,SeriesInstanceUID_anon,SOPInstanceUID_anon,img_height,img_width,laterality,view_position,horizontal_flip,weight_bearing,inverted,arthroplasty,L_KLG_inference,R_KLG_inference,SeriesDescription,StudyDescription,StudyDate_anon,age_at_exam,dicom_path
0,10001182,1.2.849.113972.3.57.1.62631357.20201023.1122916,1.2.840.113570.163246170217.20201019125748437223,1.2.826.0.1.3680043.8.498.40281381358579896373...,2539.0,2079.0,R,L,0.0,1.0,0,0,NaN,NaN,LATERAL,XR Knee 3 Views Right Standard Protocol,2020-05-24,62,10001182/1.2.849.113972.3.57.1.62631357.202010...
1,10001758,1.2.840.113975.3.59.1.64324621.20210625.1133227,1.2.842.113572.163246212233.20210618135511646926,1.2.826.0.1.3680043.8.498.73756677603175613024...,2589.0,1820.0,R,L,0.0,0.0,0,0,NaN,NaN,Lateral,XR Knee 3 Views Right Standard Protocol,2021-08-06,41,10001758/1.2.840.113975.3.59.1.64324621.202106...
2,10002144,1.2.846.113977.3.58.1.56999138.20180517.1095048,1.2.846.113622.2.70.2207549553.205161805101005...,1.2.826.0.1.3680043.8.498.13086143241040152779...,1774.0,1045.0,R,I,0.0,0.0,0,0,NaN,NaN,XR Knee 3 Views Bilateral,XR KNEE 3 VIEWS BILATERAL,2018-02-13,40,10002144/1.2.846.113977.3.58.1.56999138.201805...
3,10002144,1.2.846.113977.3.58.1.56999138.20180517.1095048,1.2.846.113622.2.70.2207549553.205161805101005...,1.2.826.0.1.3680043.8.498.13222315862241217925...,1774.0,1045.0,R,F,0.0,0.0,0,0,NaN,NaN,XR Knee 3 Views Bilateral,XR KNEE 3 VIEWS BILATERAL,2018-02-13,40,10002144/1.2.846.113977.3.58.1.56999138.201805...
4,10002144,1.2.846.113977.3.58.1.56999138.20180517.1095048,1.2.846.113622.2.70.2207549553.205161805101005...,1.2.826.0.1.3680043.8.498.15405157314397326116...,1774.0,1044.0,L,F,0.0,0.0,0,0,NaN,NaN,XR Knee 3 Views Bilateral,XR KNEE 3 VIEWS BILATERAL,2018-02-13,40,10002144/1.2.846.113977.3.58.1.56999138.201805...


In [18]:
master_df = pd.merge(img_df, demo_df, how='left', on='empi_anon')

In [36]:
lateral_master_df = master_df[master_df.view_position == 'L']

In [37]:
lateral_master_df.head()

,empi_anon,StudyInstanceUID_anon,SeriesInstanceUID_anon,SOPInstanceUID_anon,img_height,img_width,laterality,view_position,horizontal_flip,weight_bearing,...,L_KLG_inference,R_KLG_inference,SeriesDescription,StudyDescription,StudyDate_anon,age_at_exam,dicom_path,sex,race,ethnicity
0,10001182,1.2.849.113972.3.57.1.62631357.20201023.1122916,1.2.840.113570.163246170217.20201019125748437223,1.2.826.0.1.3680043.8.498.40281381358579896373...,2539.0,2079.0,R,L,0.0,1.0,...,NaN,NaN,LATERAL,XR Knee 3 Views Right Standard Protocol,2020-05-24,62,10001182/1.2.849.113972.3.57.1.62631357.202010...,Male,Unknown,Unknown
1,10001758,1.2.840.113975.3.59.1.64324621.20210625.1133227,1.2.842.113572.163246212233.20210618135511646926,1.2.826.0.1.3680043.8.498.73756677603175613024...,2589.0,1820.0,R,L,0.0,0.0,...,NaN,NaN,Lateral,XR Knee 3 Views Right Standard Protocol,2021-08-06,41,10001758/1.2.840.113975.3.59.1.64324621.202106...,Female,Multiple,Unknown
6,10002144,1.2.846.113977.3.58.1.56999138.20180517.1095048,1.2.846.113622.2.70.2207549553.205161805101005...,1.2.826.0.1.3680043.8.498.25097243441976964330...,1775.0,1059.0,R,L,0.0,0.0,...,NaN,NaN,XR Knee 3 Views Bilateral,XR KNEE 3 VIEWS BILATERAL,2018-02-13,40,10002144/1.2.846.113977.3.58.1.56999138.201805...,Female,Caucasian or White,Non-Hispanic or Latino
7,10002144,1.2.846.113977.3.58.1.56999138.20180517.1095048,1.2.846.113622.2.70.2207549553.205161805101005...,1.2.826.0.1.3680043.8.498.73980548070501957600...,1771.0,1058.0,L,L,0.0,0.0,...,NaN,NaN,XR Knee 3 Views Bilateral,XR KNEE 3 VIEWS BILATERAL,2018-02-13,40,10002144/1.2.846.113977.3.58.1.56999138.201805...,Female,Caucasian or White,Non-Hispanic or Latino
9,10003180,1.2.845.113979.3.64.1.55016411.20170539.1,1.2.846.113564.1632467445.20170530081259855397,1.2.826.0.1.3680043.8.498.12678074990125916447...,2517.0,1985.0,R,L,0.0,1.0,...,NaN,NaN,LATERAL,XR Knee 3 Views Right Standard Protocol,2017-11-09,67,10003180/1.2.845.113979.3.64.1.55016411.201705...,Female,Caucasian or White,Non-Hispanic or Latino


### Setup (once)

1. Run Cell 1 — define paths, add mask_path column to lateral_master_df

### Each segmentation session (repeat until done)

2. Run Cell 2 — check status (how many segmented vs not)
3. Run Cell 3 — copy next batch of unsegmented DICOMs to SSD
4. Open 3D Slicer → segment tibia, femur, patella → save each mask as <stem>.nrrd to SSD/masks/
5. Run Cell 4 — copy masks HDD, verify, delete from SSD, refresh is_segmented

### Once enough masks collected (20–100)

6. Train segmentation model on annotated subset
7. Run inference on remaining 183k images → pseudo-labels
8. (Optional) Active learning — correct uncertain predictions, retrain

### Per image downstream pipeline

9. Load DICOM + mask via KneeDataset
10. Solve Laplace/Eikonal on tibia mask → intrinsic coordinate field
11. Compute posterior tibial slope from coordinate system
12. Store results back to lateral_master_df

In [41]:
# Cell 1

import shutil, os
import pandas as pd


SSD_path = '../../../../../Volumes/SSD_01/'
mrkr_data_path ='../../../../../Volumes/HDD_02/datasets/emory_mrkr/'

mrkr_data_path = os.path.abspath(mrkr_data_path)
SSD_path = os.path.abspath(SSD_path)

SSD_xray_path = os.path.join(SSD_path,'knee-landmarks-xray','temp')


SSD_dicom_dir = os.path.join(SSD_xray_path, 'images')
SSD_mask_dir  = os.path.join(SSD_xray_path, 'masks')
HDD_mask_dir  = os.path.join(mrkr_data_path, 'masks')

os.makedirs(SSD_dicom_dir, exist_ok=True)
os.makedirs(SSD_mask_dir,  exist_ok=True)
os.makedirs(HDD_mask_dir,  exist_ok=True)

def build_mask_path(dicom_path):
    stem = os.path.splitext(os.path.basename(dicom_path))[0]
    return os.path.join(HDD_mask_dir, f"{stem}.nrrd")


mrkr_tables_path = os.path.join(mrkr_data_path,'tables')

table_names = os.listdir(mrkr_tables_path)
table_names = os.listdir(mrkr_tables_path)

image_df = pd.read_csv(
    os.path.join(
        mrkr_tables_path,
        table_names[-3]
    )
)

demo_df = pd.read_csv(
    os.path.join(
        mrkr_tables_path,
        table_names[-4]
    )
)

lateral_master_csv = os.path.join(mrkr_data_path, 'tables', 'mrkrk_lateral_image_metadata.csv')

if os.path.exists(lateral_master_csv):
    lateral_master_df = pd.read_csv(lateral_master_csv)
else:
    # build from image_df as before
    lateral_master_df = image_df[image_df.view_position == 'L'].reset_index(drop=True)
    lateral_master_df['mask_path'] = lateral_master_df['dicom_path'].apply(build_mask_path)
    lateral_master_df['is_segmented'] = False
    lateral_master_df.to_csv(lateral_master_csv, index=False)


lateral_master_df['mask_path'] = lateral_master_df['dicom_path'].apply(build_mask_path)

lateral_master_df.to_csv(os.path.join(mrkr_data_path, 'tables', 'mrkrk_lateral_image_metadata.csv'), index=False)

In [42]:
# Skip check if no masks directory exists yet or it's empty
if not os.path.exists(HDD_mask_dir) or not os.listdir(HDD_mask_dir):
    lateral_master_df['is_segmented'] = False
    print("Segmented:   0")
    print(f"Unsegmented: {len(lateral_master_df)}")
else:
    lateral_master_df['is_segmented'] = lateral_master_df['mask_path'].apply(os.path.exists)
    print(f"Segmented:   {lateral_master_df['is_segmented'].sum()}")
    print(f"Unsegmented: {(~lateral_master_df['is_segmented']).sum()}")


Segmented:   0
Unsegmented: 195802


In [45]:
# Cell 3

# Pick unsegmented images to annotate next
# Replace .head(20) with any filter (e.g. stratified by patient, age, etc.)

# Sample diverse patients rather than first N rows
to_segment = (
    lateral_master_df[~lateral_master_df['is_segmented']]
    .groupby(['empi_anon', 'laterality'], group_keys=False)
    .apply(lambda g: g.sample(1))
    .sample(n=20, random_state=42)
    .reset_index(drop=True)
)

for _, row in to_segment.iterrows():
    src = os.path.join(mrkr_data_path, 'images', row['dicom_path'])
    dst = os.path.join(SSD_dicom_dir, os.path.basename(row['dicom_path']))
    if not os.path.exists(dst):
        shutil.copy2(src, dst)

print(f"Copied {len(to_segment)} DICOMs to SSD")
print(f"Open in 3D Slicer: {SSD_dicom_dir}")
print(f"Save masks to:     {SSD_mask_dir}")


/var/folders/t2/tc59j3yx5mb4thryyt13twhw0000gn/T/ipykernel_65187/1014006067.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(1))


Copied 20 DICOMs to SSD
Open in 3D Slicer: /Volumes/SSD_01/knee-landmarks-xray/temp/images
Save masks to:     /Volumes/SSD_01/knee-landmarks-xray/temp/masks


In [ ]:
# Cell 4

synced, missing = [], []

for _, row in to_segment.iterrows():
    stem = os.path.splitext(os.path.basename(row['dicom_path']))[0]
    src_mask  = os.path.join(SSD_mask_dir,  f"{stem}.nrrd")
    src_dicom = os.path.join(SSD_dicom_dir, os.path.basename(row['dicom_path']))
    dst_mask  = row['mask_path']

    if os.path.exists(src_mask):
        shutil.copy2(src_mask, dst_mask)

        if os.path.exists(dst_mask):
            os.remove(src_mask)
            os.remove(src_dicom)
            synced.append(stem)
    else:
        missing.append(stem)

lateral_master_df['is_segmented'] = lateral_master_df['mask_path'].apply(os.path.exists)

print(f"Synced and cleaned: {len(synced)}")
print(f"Still needs segmentation: {len(missing)}")
if missing:
    print(missing)


lateral_master_df.to_csv(os.path.join(mrkr_data_path, 'tables', 'MRKR_lateral_image_metadata.csv'), index=False)


In [43]:
lateral_master_df.head()

,empi_anon,StudyInstanceUID_anon,SeriesInstanceUID_anon,SOPInstanceUID_anon,img_height,img_width,laterality,view_position,horizontal_flip,weight_bearing,...,arthroplasty,L_KLG_inference,R_KLG_inference,SeriesDescription,StudyDescription,StudyDate_anon,age_at_exam,dicom_path,mask_path,is_segmented
0,10001182,1.2.849.113972.3.57.1.62631357.20201023.1122916,1.2.840.113570.163246170217.20201019125748437223,1.2.826.0.1.3680043.8.498.40281381358579896373...,2539.0,2079.0,R,L,0.0,1.0,...,0,NaN,NaN,LATERAL,XR Knee 3 Views Right Standard Protocol,2020-05-24,62,10001182/1.2.849.113972.3.57.1.62631357.202010...,/Volumes/HDD_02/datasets/emory_mrkr/masks/1.2....,False
1,10001758,1.2.840.113975.3.59.1.64324621.20210625.1133227,1.2.842.113572.163246212233.20210618135511646926,1.2.826.0.1.3680043.8.498.73756677603175613024...,2589.0,1820.0,R,L,0.0,0.0,...,0,NaN,NaN,Lateral,XR Knee 3 Views Right Standard Protocol,2021-08-06,41,10001758/1.2.840.113975.3.59.1.64324621.202106...,/Volumes/HDD_02/datasets/emory_mrkr/masks/1.2....,False
2,10002144,1.2.846.113977.3.58.1.56999138.20180517.1095048,1.2.846.113622.2.70.2207549553.205161805101005...,1.2.826.0.1.3680043.8.498.25097243441976964330...,1775.0,1059.0,R,L,0.0,0.0,...,0,NaN,NaN,XR Knee 3 Views Bilateral,XR KNEE 3 VIEWS BILATERAL,2018-02-13,40,10002144/1.2.846.113977.3.58.1.56999138.201805...,/Volumes/HDD_02/datasets/emory_mrkr/masks/1.2....,False
3,10002144,1.2.846.113977.3.58.1.56999138.20180517.1095048,1.2.846.113622.2.70.2207549553.205161805101005...,1.2.826.0.1.3680043.8.498.73980548070501957600...,1771.0,1058.0,L,L,0.0,0.0,...,0,NaN,NaN,XR Knee 3 Views Bilateral,XR KNEE 3 VIEWS BILATERAL,2018-02-13,40,10002144/1.2.846.113977.3.58.1.56999138.201805...,/Volumes/HDD_02/datasets/emory_mrkr/masks/1.2....,False
4,10003180,1.2.845.113979.3.64.1.55016411.20170539.1,1.2.846.113564.1632467445.20170530081259855397,1.2.826.0.1.3680043.8.498.12678074990125916447...,2517.0,1985.0,R,L,0.0,1.0,...,R,NaN,NaN,LATERAL,XR Knee 3 Views Right Standard Protocol,2017-11-09,67,10003180/1.2.845.113979.3.64.1.55016411.201705...,/Volumes/HDD_02/datasets/emory_mrkr/masks/1.2....,False
